# Exp 31: Fixed thresholds

Card: [exp31_fixed_thresholds.md](../../experiments/exp31_fixed_thresholds.md)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from core.config import CLASSES, ROOT
from core.cv import VIP, Config, CVResult
from core.predict import best_f1

CONFIG_DIR = ROOT / "sections" / "model" / "configs"
MODELS = {
    "exp11": "exp11_resnet18_5fold",
    "exp29": "exp29_downsample_5fold",
    "exp26": "exp26_coarse_dropout_5fold",
}
# hand thresholds per model, edit here
THRESHOLDS = {
    "exp11": {"Cat": 0.5, "Dog": 0.5, "Moris": 0.9, "Motya": 0.9, "Biatrix": 0.7},
    "exp29": {"Cat": 0.5, "Dog": 0.5, "Moris": 0.9, "Motya": 0.9, "Biatrix": 0.7},
    "exp26": {"Cat": 0.5, "Dog": 0.5, "Moris": 0.9, "Motya": 0.9, "Biatrix": 0.7},
}

## Load OOF predictions


In [ ]:
results = {
    name: CVResult.load(Config.from_toml(CONFIG_DIR / slug / "resnet18.toml"))
    for name, slug in MODELS.items()
}
# pooled OOF rows of folds 0-4, best-F1 thresholds as in submission
best = {name: best_f1(r.labels, r.probs)["threshold"] for name, r in results.items()}
hand = {name: pd.Series(THRESHOLDS[name])[CLASSES] for name in results}

## F1 per class and Macro F1


In [ ]:
def f1_row(r, thresholds):
    f1 = f1_score(
        r.labels, r.probs >= thresholds.to_numpy(), average=None, zero_division=0
    )
    return pd.Series(np.append(f1, f1.mean()), index=[*CLASSES, "Macro F1"])


f1 = pd.concat(
    {
        name: pd.DataFrame(
            {"best": f1_row(r, best[name]), "hand": f1_row(r, hand[name])}
        ).T
        for name, r in results.items()
    }
)
f1.round(3)

## TP / FP / FN, VIP


In [ ]:
def counts(r, thresholds):
    y, pred = r.labels == 1, r.probs >= thresholds.to_numpy()
    df = pd.DataFrame(
        {"TP": (y & pred).sum(0), "FP": (~y & pred).sum(0), "FN": (y & ~pred).sum(0)},
        index=CLASSES,
    )
    return df.loc[VIP]


for name, r in results.items():
    table = pd.concat(
        {"best": counts(r, best[name]), "hand": counts(r, hand[name])}, axis=1
    ).swaplevel(axis=1)
    display(table[["TP", "FP", "FN"]].rename_axis(name))